# MongoDB Replica Set

Un espacio para **jugar** con el replica set `rs0`: escribir y ver la copia en cada nodo, leer el oplog, medir
write concerns, decidir a qué nodo va cada lectura, provocar un failover, cambiar prioridades y escuchar change
streams. Las lecciones (`lessons/`) muestran cada tema en orden; aquí puedes cambiar cualquier cosa y volver a
ejecutar.

**Cómo usarlo**

1. Ejecuta las secciones 0 y 1 (configuración y herramientas) una vez.
2. Después salta a la sección que quieras. Las celdas marcadas con **Juega aquí** están hechas para que las cambies.
3. Si algo queda raro, `python lessons/00_seed.py` deja la base `company` como al principio. La sección 11 limpia
   lo que crea este notebook.

> Requisito: haber ejecutado `python lessons/00_seed.py` al menos una vez.

| # | Sección | Qué puedes probar |
| --- | --- | --- |
| 0 | Configuración | Conexión al replica set |
| 1 | Herramientas | `miembros`, `en_cada_nodo`, `oplog`, `plan`, `tabla` |
| 2 | Explorar la colección | `find`, `aggregate`, proyecciones |
| 3 | CRUD y replicación | Escribir en el PRIMARY y ver la copia en cada nodo y en el oplog |
| 4 | Índices y `explain` | `COLLSCAN` vs `IXSCAN` con tus propios filtros |
| 5 | Agregaciones | Tu propio pipeline |
| 6 | Anatomía del replica set | Roles, retraso, oplog y su ventana |
| 7 | Prioridades | Cambiar quién es el PRIMARY preferido |
| 8 | Consistencia | Write concern, read preference, read concern, sesiones causales |
| 9 | Failover | El PRIMARY renuncia mientras escribes; observar mientras apagas un nodo |
| 10 | Change streams | Escuchar cambios en tiempo real y reanudar con el resume token |
| 11 | Limpieza | Borrar lo creado aquí |

## 0. Configuración

La conexión sale de `MONGO_URI` (`.envs/mongo.env`): `mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=rs0`.
El driver recibe la lista de nodos y el nombre del set, pregunta a cada uno su rol (`hello`) y envía las escrituras
al PRIMARY. El notebook usa las mismas funciones que las lecciones (`src/lab/`).

In [ ]:
import threading
import time
from collections import Counter
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
from pymongo import MongoClient, ReadPreference, ReturnDocument, WriteConcern
from pymongo.errors import DuplicateKeyError, PyMongoError, WriteConcernError
from pymongo.read_concern import ReadConcern

from lab.client import get_client
from lab.config import MONGO_DB, MONGO_URI

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

client = get_client()
db = client[MONGO_DB]
employees = db.employees

print("URI:       ", MONGO_URI)
print("MongoDB:   ", client.server_info()["version"])
print("PRIMARY:   ", client.primary)
print("SECONDARY: ", sorted(client.secondaries))
print("Base:      ", MONGO_DB, "| colecciones:", db.list_collection_names())
print("Empleados: ", employees.estimated_document_count())

## 1. Herramientas

Funciones cortas, a la vista, para que las leas y las cambies.

| Función | Qué hace |
| --- | --- |
| `tabla(cursor_o_lista)` | Convierte resultados de MongoDB en un DataFrame |
| `miembros()` | Estado de cada nodo: rol, salud, prioridad, votos, retraso y de quién copia |
| `nodo(host)` | Conexión **directa** a un nodo (`directConnection=true`), sin pasar por el replica set |
| `en_cada_nodo(filtro)` | Lee el mismo documento directamente en mongo1, mongo2 y mongo3 |
| `oplog(n, ns)` | Las últimas `n` operaciones del oplog |
| `plan(filtro)` | El plan que elige MongoDB (`explain`): etapas, documentos revisados y devueltos |
| `observar(segundos)` | Imprime el rol de cada nodo cada 2 s (útil mientras apagas contenedores) |

In [ ]:
NODOS = ["mongo1:27017", "mongo2:27017", "mongo3:27017"]
_conexiones = {}


def tabla(rows):
    return pd.DataFrame(list(rows))


def miembros():
    status = client.admin.command("replSetGetStatus")
    config = {m["host"]: m for m in client.admin.command("replSetGetConfig")["config"]["members"]}
    primary = next((m.get("optimeDate") for m in status["members"] if m["stateStr"] == "PRIMARY"), None)
    return pd.DataFrame([{
        "nodo": m["name"],
        "estado": m["stateStr"],
        "salud": m["health"],
        "prioridad": config[m["name"]]["priority"],
        "votos": config[m["name"]]["votes"],
        # aproximado: sin escrituras, el PRIMARY solo anota un no-op cada 10 s y el retraso puede verse de hasta ~10 s
        "retraso_s": (primary - m["optimeDate"]).total_seconds() if primary and m.get("optimeDate") else None,
        "copia_de": m.get("syncSourceHost") or "-",
    } for m in status["members"]])


def nodo(host):
    """Conexión directa a un nodo: lee lo que tiene ESE nodo, sea PRIMARY o SECONDARY."""
    if host not in _conexiones:
        _conexiones[host] = MongoClient(f"mongodb://{host}/?directConnection=true", tz_aware=True,
                                        serverSelectionTimeoutMS=2000)
    return _conexiones[host]


def en_cada_nodo(filtro, proyeccion=None, coleccion="employees"):
    filas = []
    for host in NODOS:
        try:
            n = nodo(host)
            rol = "PRIMARY" if n.admin.command("hello")["isWritablePrimary"] else "SECONDARY"
            filas.append({"nodo": host, "rol": rol, "documento": n[MONGO_DB][coleccion].find_one(filtro, proyeccion)})
        except PyMongoError as error:
            filas.append({"nodo": host, "rol": "sin conexión", "documento": type(error).__name__})
    return pd.DataFrame(filas)


def oplog(n=5, ns=f"{MONGO_DB}.employees"):
    filtro = {"ns": ns} if ns else {}
    return pd.DataFrame([{"hora": e["wall"].strftime("%H:%M:%S.%f")[:-3], "op": e["op"], "ns": e["ns"],
                          "cambio": str(e.get("o", {}))[:90]}
                         for e in client.local["oplog.rs"].find(filtro).sort("$natural", -1).limit(n)])


def plan(filtro, coleccion="employees"):
    explain = db.command("explain", {"find": coleccion, "filter": filtro}, verbosity="executionStats")
    etapas, ganador = [], explain["queryPlanner"]["winningPlan"]
    ganador = ganador.get("queryPlan", ganador)
    while ganador:
        etapas.append(ganador["stage"] + (f"({ganador['indexName']})" if "indexName" in ganador else ""))
        ganador = ganador.get("inputStage")
    stats = explain["executionStats"]
    return {"plan": " <- ".join(etapas), "devueltos": stats["nReturned"],
            "documentos_revisados": stats["totalDocsExamined"], "claves_revisadas": stats["totalKeysExamined"],
            "ms": stats["executionTimeMillis"]}


def observar(segundos=30, cada=2):
    fin = time.time() + segundos
    while time.time() < fin:
        try:
            estados = {m["name"].split(":")[0]: m["stateStr"]
                       for m in client.admin.command("replSetGetStatus")["members"]}
        except PyMongoError as error:
            estados = {"error": type(error).__name__}
        print(datetime.now().strftime("%H:%M:%S"), " | ".join(f"{k}: {v}" for k, v in estados.items()))
        time.sleep(cada)


miembros()

## 2. Explorar la colección

In [ ]:
# Un empleado
employees.find_one({"employee_id": "E0001"}, {"_id": 0})

### Juega aquí: cualquier `find`

In [ ]:
filtro = {"department": "Ingeniería", "city": "Medellín", "salary": {"$gte": 10_000_000}}   # cambia el filtro
proyeccion = {"_id": 0, "employee_id": 1, "name": 1, "title": 1, "salary": 1, "skills": 1}
print(employees.count_documents(filtro), "empleados")
tabla(employees.find(filtro, proyeccion).sort("salary", -1).limit(10))

# Ideas:
#   {"skills": {"$all": ["python", "aws"]}}             arrays que contienen todos los valores
#   {"skills": {"$size": 1}}                            arrays de un solo elemento
#   {"$or": [{"city": "Cali"}, {"salary": {"$lt": 3_000_000}}]}
#   {"name": {"$regex": "^and", "$options": "i"}}
#   {"active": False}

## 3. CRUD y replicación

Toda escritura va al PRIMARY, queda en su oplog y los SECONDARY la copian. Aquí escribes un empleado propio y lo
lees **directamente** en cada nodo.

### Juega aquí: tu empleado

In [ ]:
mi_empleado = {
    "employee_id": "E9500",
    "name": "Tu Nombre",
    "email": "tu.nombre@company.com",
    "department": "Ingeniería",
    "title": "Desarrollador",
    "city": "Bogotá",
    "salary": 7_000_000,
    "skills": ["python", "mongodb"],
    "hired_at": datetime.now(timezone.utc),
    "active": True,
}
employees.delete_many({"employee_id": "E9500"})
print("insert_one ->", employees.insert_one(mi_empleado).inserted_id)
en_cada_nodo({"employee_id": "E9500"}, {"_id": 0, "name": 1, "salary": 1, "skills": 1})

In [ ]:
# Update: cambia los operadores ($set, $inc, $push, $addToSet, $unset, $rename...) y vuelve a ejecutar
cambio = {"$inc": {"salary": 500_000}, "$addToSet": {"skills": "docker"}, "$set": {"title": "Tech Lead"}}
print(employees.find_one_and_update({"employee_id": "E9500"}, cambio, projection={"_id": 0, "title": 1, "salary": 1, "skills": 1},
                                    return_document=ReturnDocument.AFTER))
en_cada_nodo({"employee_id": "E9500"}, {"_id": 0, "title": 1, "salary": 1, "skills": 1})

In [ ]:
# El oplog guarda el cambio ya resuelto: un $inc queda como el valor final ($set), así es idempotente
oplog(4)

In [ ]:
# El índice único de email rechaza duplicados (también en los SECONDARY, que aplican lo mismo)
try:
    employees.insert_one({"employee_id": "E9501", "email": "tu.nombre@company.com"})
except DuplicateKeyError as error:
    print("DuplicateKeyError:", error.details["keyValue"])

In [ ]:
# Escribir directamente en un SECONDARY: no está permitido
secundario = next(h for h in NODOS if not nodo(h).admin.command("hello")["isWritablePrimary"])
try:
    nodo(secundario)[MONGO_DB].employees.insert_one({"employee_id": "E9502"})
except PyMongoError as error:
    print(f"{secundario} ->", type(error).__name__, "-", str(error)[:80])

## 4. Índices y `explain`

Sin índice, MongoDB revisa todos los documentos (`COLLSCAN`). Con índice, revisa solo las entradas que coinciden
(`IXSCAN`) y luego trae esos documentos (`FETCH`).

In [ ]:
tabla({"índice": nombre, "campos": str(info["key"]), "único": info.get("unique", False)}
      for nombre, info in employees.index_information().items())

### Juega aquí: el mismo filtro con y sin índice

In [ ]:
filtro = {"city": "Cali", "salary": {"$gt": 15_000_000}}            # cambia el filtro
indice = [("city", 1), ("salary", -1)]                              # y el índice (prueba el orden de los campos)
nombre = "sandbox_idx"

if nombre in employees.index_information():
    employees.drop_index(nombre)
sin = plan(filtro)
employees.create_index(indice, name=nombre)
con = plan(filtro)
employees.drop_index(nombre)
pd.DataFrame([sin, con], index=["sin índice", "con índice"])

## 5. Agregaciones

### Juega aquí: tu pipeline

In [ ]:
pipeline = [
    {"$match": {"active": True}},
    {"$unwind": "$skills"},
    {"$group": {"_id": {"departamento": "$department", "skill": "$skills"}, "empleados": {"$sum": 1},
                "salario_promedio": {"$avg": "$salary"}}},
    {"$sort": {"empleados": -1}},
    {"$limit": 10},
]
tabla(employees.aggregate(pipeline))

# Ideas:
#   {"$bucket": {"groupBy": "$salary", "boundaries": [0, 5e6, 10e6, 20e6], "output": {"n": {"$sum": 1}}}}
#   {"$group": {"_id": {"$year": "$hired_at"}, "contratados": {"$sum": 1}}}
#   {"$facet": {"por_ciudad": [{"$sortByCount": "$city"}], "por_cargo": [{"$sortByCount": "$title"}]}}

## 6. Anatomía del replica set

In [ ]:
# Lo que el driver pregunta para descubrir la topología
hello = client.admin.command("hello")
{k: hello[k] for k in ("setName", "hosts", "primary", "me", "isWritablePrimary", "electionId")}

In [ ]:
miembros()

In [ ]:
# Ventana del oplog: cuánto tiempo de historia guarda. Un SECONDARY apagado más tiempo que esto
# ya no puede ponerse al día copiando el oplog y necesita una resincronización completa.
primero, ultimo = (client.local["oplog.rs"].find_one(sort=[("$natural", d)])["wall"] for d in (1, -1))
stats = client.local.command("collStats", "oplog.rs")
print(f"desde {primero:%Y-%m-%d %H:%M:%S} hasta {ultimo:%Y-%m-%d %H:%M:%S}  "
      f"({(ultimo - primero).total_seconds() / 3600:.1f} h, {stats['size'] / 1024 / 1024:.1f} MB "
      f"de {stats['maxSize'] / 1024 / 1024:.0f} MB)")

### Juega aquí: el oplog

In [ ]:
# Cambia ns (None = todas las colecciones) y n. op: i = insert, u = update, d = delete, c = comando, n = no-op
oplog(n=10, ns=None)

## 7. Prioridades

`mongo1` tiene prioridad 2 y los demás 1, así que es el PRIMARY preferido. Si subes la prioridad de otro nodo por
encima, ese nodo pide una elección en cuanto está al día (*priority takeover*) y pasa a ser PRIMARY.

### Juega aquí: cambia el PRIMARY preferido

In [ ]:
def prioridades(nuevas):
    config = client.admin.command("replSetGetConfig")["config"]
    for m in config["members"]:
        m["priority"] = nuevas.get(m["host"], m["priority"])
    config["version"] += 1
    client.admin.command("replSetReconfig", config)


prioridades({"mongo1:27017": 1, "mongo2:27017": 3, "mongo3:27017": 1})     # prueba con mongo3
for _ in range(30):
    time.sleep(1)
    if client.admin.command("hello").get("primary") == "mongo2:27017":
        break
miembros()

In [ ]:
# Restaurar la configuración original: mongo1 vuelve a ser el PRIMARY preferido
prioridades({"mongo1:27017": 2, "mongo2:27017": 1, "mongo3:27017": 1})
for _ in range(30):
    time.sleep(1)
    if client.admin.command("hello").get("primary") == "mongo1:27017":
        break
miembros()

## 8. Consistencia

| Opción | Qué decide |
| --- | --- |
| **Write concern** | Cuántos nodos confirman una escritura antes de responder |
| **Read preference** | A qué nodo va una lectura |
| **Read concern** | Qué tan confirmados deben estar los datos leídos |

### Juega aquí: write concern

In [ ]:
def latencia(concern, n=100):
    coleccion = employees.with_options(write_concern=concern)
    inicio = time.perf_counter()
    for i in range(n):
        coleccion.update_one({"employee_id": "E0002"}, {"$set": {"contador": i}})
    return (time.perf_counter() - inicio) * 1000 / n


niveles = {                                           # agrega o quita niveles
    "w=0 (sin confirmación)": WriteConcern(w=0),
    "w=1 (solo PRIMARY)": WriteConcern(w=1),
    "w=majority (2 de 3)": WriteConcern(w="majority"),
    "w=3 (todos)": WriteConcern(w=3, wtimeout=5000),
    "w=majority + journal": WriteConcern(w="majority", j=True),
}
pd.DataFrame([{"write concern": k, "ms por escritura": round(latencia(v), 2)} for k, v in niveles.items()])

In [ ]:
# Un write concern imposible: w=4 con 3 nodos (wtimeout para no esperar para siempre)
try:
    employees.with_options(write_concern=WriteConcern(w=4, wtimeout=2000)).update_one(
        {"employee_id": "E0002"}, {"$set": {"x": 1}})
except WriteConcernError as error:
    print("WriteConcernError:", error.details.get("codeName"), "- la escritura SÍ se aplicó, solo no se confirmó a 4 nodos")

### Juega aquí: read preference

In [ ]:
modos = [ReadPreference.PRIMARY, ReadPreference.PRIMARY_PREFERRED, ReadPreference.SECONDARY,
         ReadPreference.SECONDARY_PREFERRED, ReadPreference.NEAREST]
filas = []
for modo in modos:
    atendido_por = Counter(db.command("hello", read_preference=modo)["me"] for _ in range(30))
    filas.append({"read preference": modo.name, **{h: atendido_por.get(h, 0) for h in NODOS}})
pd.DataFrame(filas).set_index("read preference")

In [ ]:
# Read concern + sesión causal: leer tus propias escrituras desde un SECONDARY
with client.start_session(causal_consistency=True) as sesion:
    employees.update_one({"employee_id": "E0002"}, {"$set": {"contador": 999}}, session=sesion)
    secundario = employees.with_options(read_preference=ReadPreference.SECONDARY, read_concern=ReadConcern("majority"))
    print("leído en un SECONDARY:", secundario.find_one({"employee_id": "E0002"}, {"_id": 0, "contador": 1}, session=sesion))

for nivel in ("local", "majority", "linearizable"):
    coleccion = employees.with_options(read_concern=ReadConcern(nivel))
    inicio = time.perf_counter()
    coleccion.find_one({"employee_id": "E0002"})
    print(f"read concern {nivel:13} -> {(time.perf_counter() - inicio) * 1000:.2f} ms")

## 9. Failover

Un hilo escribe sin parar mientras el PRIMARY renuncia (`replSetStepDown`). El driver detecta el nuevo PRIMARY y
las *retryable writes* reintentan la escritura que coincide con la elección.

### Juega aquí: renuncia del PRIMARY

In [ ]:
def failover(segundos=12, renuncia_en=3, bloqueo=20):
    log = db.failover_log
    log.drop()
    alto, ticks = threading.Event(), []

    def escritor():
        segundo, ok, fallidas = int(time.time()), 0, 0
        while not alto.is_set():
            try:
                log.insert_one({"at": time.time()})
                ok += 1
            except PyMongoError:
                fallidas += 1
            if int(time.time()) != segundo:
                ticks.append({"segundo": len(ticks) + 1, "primary": client.primary and client.primary[0],
                              "ok": ok, "fallidas": fallidas})
                segundo, ok, fallidas = int(time.time()), 0, 0
            time.sleep(0.01)

    hilo = threading.Thread(target=escritor)
    hilo.start()
    time.sleep(renuncia_en)
    client.admin.command("replSetStepDown", bloqueo)       # no puede volver a ser PRIMARY durante `bloqueo` s
    time.sleep(segundos - renuncia_en)
    alto.set()
    hilo.join()
    log.drop()
    return pd.DataFrame(ticks)


resultado = failover(segundos=12, renuncia_en=3)       # cambia los tiempos
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(resultado["segundo"], resultado["ok"], label="escrituras ok")
ax.bar(resultado["segundo"], resultado["fallidas"], bottom=resultado["ok"], color="red", label="fallidas")
for _, fila in resultado.iterrows():
    ax.text(fila["segundo"], fila["ok"], fila["primary"], ha="center", va="bottom", fontsize=8, rotation=90)
ax.set_xlabel("segundo")
ax.set_ylabel("escrituras")
ax.legend()
plt.tight_layout()
resultado

### Juega aquí: apagar un nodo desde el host

Ejecuta la celda de abajo y, mientras corre, en una terminal de tu equipo:

```bash
docker stop MongoNode1      # el PRIMARY se cae de golpe: ~10 s después hay elección
docker start MongoNode1     # vuelve como SECONDARY, se pone al día y recupera el rol por prioridad
docker stop MongoNode2 MongoNode3    # sin mayoría no hay PRIMARY: nadie acepta escrituras
docker start MongoNode2 MongoNode3
```

In [ ]:
observar(segundos=60, cada=2)

## 10. Change streams

Un change stream lee el oplog y entrega cada cambio de la colección como un evento. Cada evento trae un
*resume token* para continuar donde te quedaste.

### Juega aquí: escucha y escribe

In [ ]:
def eventos(stream, maximo=20, espera=3):
    leidos, fin = [], time.time() + espera
    while len(leidos) < maximo and time.time() < fin:
        cambio = stream.try_next()
        if cambio is not None:
            leidos.append(cambio)
    return leidos


def describir(cambio):
    return {"operación": cambio["operationType"], "_id": str(cambio["documentKey"]["_id"])[-6:],
            "documento": (cambio.get("fullDocument") or {}).get("name", "-"),
            "cambios": cambio.get("updateDescription", {}).get("updatedFields", "-")}


filtro_stream = [{"$match": {"operationType": {"$in": ["insert", "update", "delete"]}}}]   # cambia el filtro
employees.delete_many({"employee_id": "E9600"})

with employees.watch(filtro_stream, full_document="updateLookup", max_await_time_ms=500) as stream:
    # Cambia estas escrituras: cada una produce un evento
    employees.insert_one({"employee_id": "E9600", "name": "Camila Reyes", "email": "camila.reyes.9600@company.com",
                          "department": "Marketing", "salary": 5_000_000})
    employees.update_one({"employee_id": "E9600"}, {"$set": {"title": "Líder de marketing"}})
    employees.update_one({"employee_id": "E9600"}, {"$inc": {"salary": 800_000}})
    employees.delete_one({"employee_id": "E9600"})
    leidos = eventos(stream, maximo=2)
    token = stream.resume_token
    display(tabla(map(describir, leidos)))

print("Reanudando con el resume token: llegan los eventos que faltaban")
with employees.watch(filtro_stream, full_document="updateLookup", resume_after=token, max_await_time_ms=500) as stream:
    display(tabla(map(describir, eventos(stream))))

# Ideas:
#   [{"$match": {"fullDocument.department": "Marketing"}}]     solo cambios de un departamento
#   [{"$match": {"updateDescription.updatedFields.salary": {"$exists": True}}}]   solo cambios de salario
#   db.watch() escucha toda la base; client.watch() todo el clúster

## 11. Limpieza

In [ ]:
print("empleados de prueba:", employees.delete_many({"employee_id": {"$in": ["E9500", "E9501", "E9502", "E9600"]}}).deleted_count)
employees.update_many({}, {"$unset": {"contador": "", "x": ""}})
db.drop_collection("failover_log")
miembros()

---

**¿Rompiste algo?** `python lessons/00_seed.py` reconstruye la base `company`. Si cambiaste prioridades, ejecuta
la celda de restauración de la sección 7.